# 04 — BUS-BRA external evaluation

This notebook evaluates a saved BUSI model on BUS-BRA. It does not train on BUS-BRA or select the classification threshold there. It prefers the `metrics.json` beside the selected checkpoint and fails if no matching BUSI validation threshold is available. A manual threshold must be explicitly supplied and must come from BUSI validation data.

## Setup

Set repository, model, archive, and output paths. The threshold is loaded from the selected checkpoint run's `metrics.json`; there is no historical default. For legacy checkpoints without matching metrics, set `BUSI_VALIDATION_THRESHOLD` explicitly to a value selected on BUSI validation data. In Colab, mount Drive only if those files are stored there. BUS-BRA files remain outside the repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/Breast-Cancer-Detection-VGG16"))
if not (PROJECT_ROOT / "src" / "evaluate.py").is_file():
    raise FileNotFoundError(f"Set PROJECT_ROOT to the repository folder: {PROJECT_ROOT}")
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
_DEFAULT_ROOT = "/content/drive/MyDrive/BreastCancerProject" if USE_GOOGLE_DRIVE else "/content/data"
MODEL_PATH = Path(os.environ.get("BUSI_MODEL", f"{_DEFAULT_ROOT}/best_model.keras"))
METRICS_PATH = Path(os.environ.get("BUSI_METRICS_JSON", str(MODEL_PATH.parent / "metrics.json")))
BUSBRA_ZIP = Path(os.environ.get("BUSBRA_ZIP", f"{_DEFAULT_ROOT}/BUSBRA.zip"))
OUTPUT_DIR = Path(os.environ.get("EXTERNAL_EVAL_OUTPUT", f"{_DEFAULT_ROOT}/busbra_external"))
BUSI_VALIDATION_THRESHOLD = os.environ.get("BUSI_VALIDATION_THRESHOLD")  # Optional explicit value; must come from BUSI validation.
RUN_EVALUATION = False  # Evaluation only; set True after confirming all paths and threshold.


## Optional external evaluation

In [ ]:
if RUN_EVALUATION:
    command = [sys.executable, "-m", "src.evaluate", "--model", str(MODEL_PATH),
               "--busbra-zip", str(BUSBRA_ZIP), "--output-dir", str(OUTPUT_DIR),
    ]
    if BUSI_VALIDATION_THRESHOLD is None:
        command.extend(["--metrics-json", str(METRICS_PATH)])
    else:
        command.extend(["--threshold", str(float(BUSI_VALIDATION_THRESHOLD))])
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Evaluation is disabled. Confirm the model and BUS-BRA paths and BUSI-derived threshold first.")
